# MumVision: veri ve görüntü işleme incelemesi
Python çekirdeği olarak projenin `.venv` ortamını seçin. Veri seti daha önce hazırlanmış olmalıdır. Gerçek piyasa OHLC verilerinden çizilen grafikler ve otomatik kural etiketleri kullanılır.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
from mumvision.dataset import ImageStore
from mumvision.detector import detect, annotate
from mumvision.patterns import PATTERNS, TURKISH
dataset = root / 'data' / 'dataset'
manifest = pd.read_csv(dataset / 'manifest.csv', keep_default_na=False)
metadata = json.loads((dataset / 'metadata.json').read_text(encoding='utf-8'))
print('Görüntü sayısı:', len(manifest))
display(manifest.groupby(['split', 'symbol']).size().unstack())

In [ ]:
row = manifest[(manifest['split'] == 'test') & manifest['labels'].str.contains('hammer')].iloc[0]
with ImageStore(dataset) as store:
    image = store.read(row)
result = detect(image)
mask = result.green_mask | result.red_mask
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(image[:, :, ::-1]); axes[0].set_title('Girdi')
axes[1].imshow(mask, cmap='gray'); axes[1].set_title('HSV renk maskesi')
axes[2].imshow(annotate(image, result)[:, :, ::-1]); axes[2].set_title('Geometri ve formasyonlar')
for axis in axes: axis.axis('off')
plt.tight_layout(); plt.show()
print('Mum sayısı:', len(result.ohlc), '| OHLC kural etiketi:', row['labels'])
display(pd.DataFrame(result.objects))

In [ ]:
counts = pd.DataFrame(metadata['last_candle_counts']).fillna(0).astype(int)
display(counts)
counts.drop(index='none', errors='ignore').plot.bar(figsize=(13, 4), logy=True)
plt.ylabel('Son mumda formasyon sayısı (log ölçek)'); plt.tight_layout(); plt.show()

In [ ]:
metrics_path = root / 'outputs' / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Test görüntüsü:', metrics['test_images'])
    display(pd.DataFrame(metrics['agreement_with_ohlc_rules']['per_class']).T)
    display(pd.DataFrame(metrics['robustness']).T[['images', 'candle_count_recovery', 'micro_f1', 'macro_f1']])
else:
    print('Önce python -m mumvision evaluate çalıştırın.')

F1, otomatik OHLC kurallarıyla uyumdur; bağımsız uzman doğruluğu veya finansal tahmin başarısı değildir. Eğitim ve test aynı ham mumları paylaşmaz, ancak aynı bölümdeki pencereler örtüşebilir. Geometri çıkarımı yalnız kırmızı/yeşil, içi dolu mumlar ve doğrusal fiyat ekseni için tasarlanmıştır. Ayrıntılar `outputs/RAPOR.md` ve `docs/FORMASYON_KURALLARI.md` dosyalarındadır.